# 🕵️‍♂️ The Ultimate Data Assessing & Cleaning Masterclass
### From Raw Data to Production-Ready Datasets

> *"Data Scientists spend 80% of their time cleaning data. The other 20% is spent complaining about cleaning data."* 

Welcome to the most comprehensive guide to Data Wrangling. This notebook combines **theoretical depth**, **production-level industry practices**, and **hands-on Pandas mastery**. 

### 🗺️ What We Will Cover:
1. **The 12-Step Learning Framework** applied to core concepts.
2. **Advanced Data Profiling** (Beyond `info()` and `describe()`).
3. **The Science of Missing Data** (MCAR, MAR, MNAR & Strategy Matrix).
4. **Outlier Detection** (IQR, Z-Score, Visualizations).
5. **Data Validation & Auditing** (Industry templates & `assert` rules).
6. **The 50+ Pandas Assessment Cheat Sheet**.
7. **Real-World Interview Scenarios** (Handling 50M+ records).
8. **End-to-End Mini Project** (Raw $\rightarrow$ Cleaned).

## 🔄 The Ultimate Data Wrangling Workflow

Before writing code, we must understand where we are in the pipeline.

```text
[ 1. ASK QUESTIONS ] ➔ What is the business problem?
         │
         ▼
[ 2. DATA WRANGLING ] ◄── WE ARE HERE
   ├─ a. Gather (CSV, API, SQL, Web Scraping)
   ├─ b. Assess (Profile, Discover, Document)
   └─ c. Clean (Define, Code, Test, Validate)
         │
         ▼
[ 3. EXPLORATORY DATA ANALYSIS (EDA) ]
         │
         ▼
[ 4. DRAW CONCLUSIONS & MODEL ]
         │
         ▼
[ 5. COMMUNICATE RESULTS ]

In [ ]:
# ==========================================
# 🛠️ ENVIRONMENT SETUP
# ==========================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Display settings for maximum readability
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.width', 1000)
pd.set_option('display.max_colwidth', 30)

print("✅ Environment Ready! Let's become Data Detectives.")

## 🔬 Module 1: Advanced Data Profiling

### 1. Theory
Data profiling is the process of examining the data available in a source to collect statistics and information about that data. It is the "fingerprinting" phase.

### 2. Why It Exists
You cannot fix what you do not understand. Profiling reveals the hidden structure, quality, and anomalies in your dataset before you attempt to clean it.

### 3. Mental Model
Think of Data Profiling like a **medical blood test**. Before a doctor prescribes a treatment (cleaning), they run a panel of tests (profiling) to check cholesterol, white blood cells, etc. `info()` is just checking the heart rate; `memory_usage()` and `nunique()` are the deep blood panel.

### 4. Diagram: The Profiling Pyramid
```text
       [ Business Context ]
      /                    \
     /   Structural (info)  \
    /------------------------\
   /   Statistical (describe) \
  /----------------------------\
 /   Granular (nunique, memory) \
/--------------------------------\
```
### 5. Real Dataset Example
We will profile a messy `patients` dataset containing demographic and health records.


In [ ]:
# ==========================================
# 📊 ADVANCED DATA PROFILING CODE
# ==========================================
# Creating a synthetic messy dataset for demonstration
np.random.seed(42)
patients = pd.DataFrame({
    'patient_id': range(1, 101),
    'age': np.random.randint(18, 90, 100),
    'weight_kg': np.random.normal(75, 15, 100).round(1),
    'state': np.random.choice(['NY', 'New York', 'CA', 'California', 'TX'], 100),
    'admission_date': pd.date_range('2023-01-01', periods=100).astype(str),
    'notes': ['Routine checkup'] * 50 + [np.nan] * 50 # 50% missing
})
# Injecting outliers
patients.loc[0, 'weight_kg'] = 250.0 
patients.loc[1, 'age'] = -5

# 6. Pandas Code: The Profiling Toolkit
print("=== 1. STRUCTURAL OVERVIEW ===")
print(patients.info(memory_usage='deep')) # Deep memory usage

print("\n=== 2. STATISTICAL OVERVIEW ===")
display(patients.describe(include='all'))

print("\n=== 3. GRANULAR PROFILING (The Missing Link) ===")
# nunique: How many unique values? (Crucial for categorical encoding later)
print("Unique values per column:\n", patients.nunique())

# memory_usage: Exact memory footprint
print("\nMemory usage (Deep):\n", patients.memory_usage(deep=True))

# value_counts: Frequency of categories (Great for spotting consistency issues)
print("\nState value counts:\n", patients['state'].value_counts())

# 7. Output Explanation
# - info() shows data types and non-null counts.
# - nunique() reveals if a 'state' column has 5 unique values (NY vs New York).
# - memory_usage(deep=True) shows actual RAM consumption (vital for big data).

# 8. Common Mistakes
# - Relying only on df.head(). You miss issues at the bottom of the dataset.
# - Using memory_usage() without deep=True (it underestimates object/string memory).

### 9. Interview Questions
**Q: Why is `nunique()` important before training a Machine Learning model?**
> **A:** It helps identify target leakage (if a column has the same nunique count as the target) and determines whether a categorical variable should be One-Hot Encoded (low nunique) or Target/Embedding Encoded (high nunique).

**Q: How do you profile a dataset that is too large to fit in RAM?**
> **A:** You use Dask or PySpark, or you profile it in chunks using Pandas `chunksize` in `read_csv`, aggregating the statistics (sum, count, min, max) iteratively.

### 10. Practice Tasks
- [ ] Calculate the percentage of missing values for every column.
- [ ] Find the top 3 most memory-intensive columns.

### 11. Advanced Tasks
- [ ] Write a function that returns a DataFrame summarizing: `dtype`, `missing_count`, `missing_pct`, `nunique`, and `memory_mb` for all columns.

## 🕳️ Module 2: Missing Data Mechanisms & Strategy Matrix

### 1. Theory
Missing data isn't just "empty cells". Statisticians classify missing data into three distinct mechanisms. How you clean it depends entirely on *why* it is missing.

### 2. Why It Exists
Imputing data incorrectly introduces bias. If you fill missing "income" values with the median, but the data is missing *because* high-income people refuse to share, you've ruined the distribution.

### 3. Mental Model
*   **MCAR (Missing Completely At Random):** The data went on vacation. It has nothing to do with the patient. (e.g., A sensor randomly glitched).
*   **MAR (Missing At Random):** The missingness is related to *other* observed variables. (e.g., Men are less likely to fill out a "depression scale" than women. It depends on Gender, not the depression score itself).
*   **MNAR (Missing Not At Random):** The missingness is related to the *missing value itself*. (e.g., Very heavy people are less likely to step on a scale. The weight is missing *because* the weight is high).

### 4. Diagram: The Strategy Matrix

| Mechanism | Severity | Strategy Matrix (When to...) |
| :--- | :--- | :--- |
| **MCAR** | Low | **Remove:** If < 5% of data. <br> **Impute:** Mean/Median/KNN if > 5%. |
| **MAR** | Medium | **Impute:** Use predictive models (Regression/KNN) using the related variables. |
| **MNAR** | High | **Flag:** Create a new "Is_Missing" column. <br> **Transform:** Use non-parametric models. Do NOT just impute mean! |

### 5. Real Dataset Example

Let's test for MCAR and apply the Strategy Matrix.


In [ ]:
# ==========================================
# 🕳️ MISSING DATA HANDLING CODE
# ==========================================
df = patients.copy()

# 6. Pandas Code: Implementing the Strategy Matrix
# Step A: Identify Missingness
missing_stats = df.isnull().sum()
print("Missing Values:\n", missing_stats[missing_stats > 0])

# Step B: Test for MCAR (Little's Test is standard, but we can use visual correlation)
# If 'notes' is MCAR, its missingness shouldn't correlate with 'age' or 'weight'
missing_indicator = df['notes'].isnull().astype(int)
print("\nCorrelation between 'notes_missing' and 'age':", 
      missing_indicator.corr(df['age'])) 
# If correlation is ~0, it's likely MCAR.

# Step C: Apply Strategy Matrix
# Strategy 1: Remove (For MCAR with < 5% missing)
# Let's say 'weight_kg' had 3% missing. We would do: df.dropna(subset=['weight_kg'])

# Strategy 2: Impute (For MAR/MCAR with > 5% missing)
# Impute 'weight_kg' with Median (robust to outliers)
median_weight = df['weight_kg'].median()
df['weight_kg_imputed'] = df['weight_kg'].fillna(median_weight)

# Strategy 3: Flag (For MNAR - Crucial Industry Practice!)
# If 'notes' is MNAR (people skip it because they are sick), the missingness IS a feature!
df['notes_is_missing'] = df['notes'].isnull().astype(int)
# Now drop the original or impute it, but keep the flag for the ML model!
df['notes'] = df['notes'].fillna('No Notes Provided')

# 7. Output Explanation
# We didn't just blindly fill NaNs. We checked correlations, imputed continuous variables 
# with medians, and created an indicator feature for potentially MNAR data.

# 8. Common Mistakes
# - Filling categorical NaNs with "Missing" without creating a binary flag.
# - Using Mean imputation on skewed data (always use Median for skewed/financial data).
# - Dropping rows with missing target variables (you must drop them, but keep rows with missing features if using models that support it like XGBoost).

### 9. Interview Questions
**Q: How do you handle a column that is 60% missing?**
**A:** First, ask the business *why* it's missing. If it's MNAR, I will convert it to a binary flag (1 if missing, 0 if not) and drop the original column, as the missingness itself holds predictive power. If it's MCAR and irrelevant to the business question, I will drop the column entirely to reduce noise.

**Q: What is the difference between `dropna()` and `fillna()`?**
**A:** `dropna()` removes observations (rows) or features (columns) containing nulls. `fillna()` replaces nulls with a specified value (mean, median, forward-fill, etc.).

### 10. Practice Tasks
- [ ] Forward fill (`ffill`) and backward fill (`bfill`) a time-series dataset.
- [ ] Use `sklearn.impute.KNNImputer` to impute missing values based on similar rows.

### 11. Advanced Tasks
- [ ] Implement Little's MCAR test using the `pymice` or custom statistical logic.

## 📊 Module 3: Outlier Detection & Handling

### 1. Theory
Outliers are data points that differ significantly from other observations. They can be natural anomalies (e.g., Bill Gates walking into a bar) or errors (e.g., Age = 200).

### 2. Why It Exists
Outliers skew statistical measures (like the Mean) and ruin Machine Learning models (especially distance-based ones like K-Means or Linear Regression).

### 3. Mental Model
Think of outliers as **black sheep**. 
*   **IQR Method:** Looks at the "herd" (the middle 50%) and defines a fence. Anyone outside the fence is an outlier. (Robust, non-parametric).
*   **Z-Score Method:** Assumes the herd forms a perfect Bell Curve. Anyone more than 3 standard deviations away is an outlier. (Sensitive to extreme extremes).

### 4. Diagram: IQR vs Z-Score
```text
      IQR Method                   Z-Score Method
  [Min]---[Q1]===Median===[Q3]---[Max]      (-3)===0===(+3)
       |<-IQR->|                             |<-Sigma->|
  Lower Fence = Q1 - 1.5*IQR         Z = (X - Mean) / Std
  Upper Fence = Q3 + 1.5*IQR         Outlier if |Z| > 3
```
### Real Dataset Example
Detecting outliers in patient weight_kg and age.


In [ ]:
# ==========================================
# 📊 OUTLIER DETECTION CODE
# ==========================================
df_outliers = patients.copy()

# 6. Pandas Code: IQR and Z-Score Implementation
# --- METHOD 1: IQR (Best for skewed data) ---
Q1 = df_outliers['weight_kg'].quantile(0.25)
Q3 = df_outliers['weight_kg'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

iqr_outliers = df_outliers[
    (df_outliers['weight_kg'] < lower_bound) | 
    (df_outliers['weight_kg'] > upper_bound)
]
print(f"IQR Outliers in Weight: {len(iqr_outliers)}")

# --- METHOD 2: Z-Score (Best for normally distributed data) ---
z_scores = np.abs(stats.zscore(df_outliers['weight_kg'].dropna()))
z_outliers = df_outliers[z_scores > 3]
print(f"Z-Score Outliers in Weight: {len(z_outliers)}")

# --- METHOD 3: Visual Detection (Boxplots) ---
plt.figure(figsize=(10, 4))
sns.boxplot(x=df_outliers['weight_kg'], color='lightblue')
plt.title("Boxplot of Patient Weight (Outliers marked as dots)")
plt.show()

# 7. Output Explanation
# The boxplot visually confirms the 250kg weight is an outlier. 
# IQR found it because 250kg is way past Q3 + 1.5*IQR.

# 8. Common Mistakes
# - Removing outliers without investigating if they are errors or genuine anomalies.
# - Using Z-score on highly skewed data (like income or house prices). Use IQR or Log-Transform instead!

### 9. Interview Questions
**Q: Should you always remove outliers?**
**A:** No! If the outlier is a data entry error (Age = 200), fix or remove it. If it's a genuine anomaly (e.g., a massive spike in server traffic during a DDoS attack), it is the most important data point! You might need to use Robust Scalers or tree-based models (Random Forest) which are immune to outliers.

**Q: How do you handle outliers in a dataset with 50 million rows?**
**A:** Calculating Z-scores on 50M rows is memory intensive. I would use approximate quantiles (like in Dask/Spark) or use the IQR method, which only requires calculating the 25th and 75th percentiles, which is highly optimized in distributed computing.

### 10. Practice Tasks
- [ ] Cap outliers using `np.clip()` instead of removing them.
- [ ] Apply a Log Transformation (`np.log1p`) to a skewed column and re-run the Z-score test.

## 🧹 Module 4: The Cleaning Framework & Validation Rules

### 1. Theory
Cleaning is not just writing code; it's an engineering process. We use the **Define $\rightarrow$ Code $\rightarrow$ Test** loop, followed by **Data Validation Rules** (`assert` statements) to guarantee the data is safe for production.

### 2. Why It Exists
Code breaks. Data changes. If you don't test your cleaning steps and assert validation rules, a bad data pipeline will silently feed garbage into your ML models (Garbage In, Garbage Out).

### 3. Mental Model
Think of **Define $\rightarrow$ Code $\rightarrow$ Test** as writing a unit test for your data. 
*   **Define:** What is the bug? (e.g., "Age cannot be negative").
*   **Code:** Fix the bug. (e.g., "Filter out negative ages").
*   **Test:** Prove the bug is dead. (e.g., `assert df['age'].min() >= 0`).

### 4. Diagram: The Golden Order of Cleaning
```text
1. COMPLETENESS (Fix missing data first, so you know your true row count)
      ↓
2. TIDINESS (Reshape data: melt, pivot, split columns)
      ↓
3. VALIDITY (Fix data types: strings to dates, objects to floats)
      ↓
4. ACCURACY (Fix factual errors: Dsvid -> David, remove impossible outliers)
      ↓
5. CONSISTENCY (Standardize formats: NY -> New York, lowercase everything)

In [ ]:
# ==========================================
# 🧹 DEFINE -> CODE -> TEST & VALIDATION
# ==========================================
df_clean = patients.copy()

# --- ISSUE: Negative Age (Accuracy/Validity) ---
# 1. DEFINE: Age cannot be negative. We will remove rows where age < 0.
print("DEFINE: Removing negative ages.")

# 2. CODE: Filter the dataframe
df_clean = df_clean[df_clean['age'] >= 0]

# 3. TEST & VALIDATION RULE: Use Python 'assert' to create a hard gate!
# If this fails, the code stops. This prevents bad data from reaching production.
assert df_clean['age'].min() >= 0, "Validation Failed: Negative ages detected!"
assert df_clean['age'].max() <= 120, "Validation Failed: Unrealistic ages detected!"
print("✅ TEST PASSED: All ages are between 0 and 120.")

# --- ISSUE: Inconsistent States (Consistency) ---
# 1. DEFINE: Map full state names to abbreviations.
state_mapping = {'New York': 'NY', 'California': 'CA'}

# 2. CODE: Apply mapping
df_clean['state'] = df_clean['state'].replace(state_mapping)

# 3. TEST & VALIDATION RULE: Assert all states are 2 letters
assert df_clean['state'].str.len().max() == 2, "Validation Failed: States not standardized!"
print("✅ TEST PASSED: All states are 2-letter abbreviations.")

# 7. Output Explanation
# The assert statements act as automated quality gates. If a future data pipeline 
# accidentally feeds in an age of -5, the notebook will crash immediately, alerting the engineer.

# 8. Common Mistakes
# - Modifying the original dataframe instead of a copy (`df_clean = df.copy()`).
# - Forgetting to test after coding. (Always check `.isnull().sum()` or `.value_counts()` after a transformation).

### 🏢 Industry Data Auditing Template
In enterprise environments, you don't just clean data; you **audit** it. Here is a standard template used by Data Stewards:

| Audit ID | Table | Column | Rule Type | Validation Logic | Status | Action Taken |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| AUD-01 | patients | age | Range | `age >= 0 & age <= 120` | ✅ Pass | Filtered 1 row |
| AUD-02 | patients | state | Regex | `^[A-Z]{2}$` | ✅ Pass | Mapped full names |
| AUD-03 | treatments | dose | Type | `isinstance(float)` | ❌ Fail | Cast to float |
| AUD-04 | patients | ssn | Format | `^\d{3}-\d{2}-\d{4}$` | ⚠️ Warn | Masked PII |

### 9. Interview Questions
**Q: How do you ensure your cleaning code doesn't break in production next month?**
**A:** I implement Data Validation Rules using tools like `Great Expectations` or `Pandera`, or simply use Python `assert` statements in the CI/CD pipeline. If the data schema changes or violates an assertion, the pipeline halts and alerts the data team.

### 10. Practice Tasks
- [ ] Write an `assert` statement to ensure a `price` column is never negative.
- [ ] Write a regex validation rule for an `email` column.

In [ ]:
# ==========================================
# 📝 THE 50+ PANDAS ASSESSMENT CHEAT SHEET
# ==========================================
# Run this cell to print the ultimate cheat sheet!

cheatsheet = """
🔍 STRUCTURAL & BASIC
1. df.shape             : (Rows, Columns)
2. df.columns           : List of column names
3. df.index             : Row indices
4. df.dtypes            : Data types of columns
5. df.info()            : Concise summary (memory, non-nulls)
6. df.head(n)           : First n rows
7. df.tail(n)           : Last n rows
8. df.sample(n)         : Random n rows

📊 STATISTICAL PROFILING
9. df.describe()        : Summary stats for numeric cols
10. df.describe(include='all') : Summary for numeric + categorical
11. df.mean()           : Mean of columns
12. df.median()         : Median of columns
13. df.std()            : Standard deviation
14. df.min() / df.max() : Min / Max values
15. df.sum()            : Sum of columns
16. df.count()          : Count of non-null values
17. df.nunique()        : Number of unique values
18. df.value_counts()   : Frequency of unique values (Categorical)
19. df.memory_usage(deep=True) : Exact memory footprint

🕳️ MISSING DATA
20. df.isnull()         : Boolean mask of missing values
21. df.isnull().sum()   : Count of missing values per col
22. df.isnull().mean()  : Percentage of missing values
23. df.dropna()         : Drop rows/cols with missing values
24. df.fillna(val)      : Fill missing values
25. df.ffill()          : Forward fill
26. df.bfill()          : Backward fill

🔁 DUPLICATES
27. df.duplicated()     : Boolean mask of duplicate rows
28. df.duplicated().sum(): Count of duplicate rows
29. df.drop_duplicates(): Remove duplicate rows

🧹 DATA CLEANING & TRANSFORMATION
30. df.rename()         : Rename columns
31. df.astype()         : Change data type
32. df.replace()        : Replace specific values
33. df.apply()          : Apply a function along an axis
34. df.map()            : Map values elementwise (Series only)
35. df.str.lower()      : String methods (lower, upper, strip)
36. pd.to_datetime()    : Convert to datetime objects
37. pd.to_numeric()     : Convert to numeric (handle errors='coerce')

🔀 RESHAPING (TIDY DATA)
38. df.melt()           : Wide to Long format
39. df.pivot()          : Long to Wide format (no aggregation)
40. df.pivot_table()    : Long to Wide (with aggregation)
41. pd.concat()         : Stack dataframes vertically/horizontally
42. df.merge()          : SQL-style joins (inner, left, right)
43. df.join()           : Join by index
44. df.explode()        : Explode lists into separate rows
45. df.transpose() / df.T : Transpose rows and columns

📈 GROUPING & AGGREGATION
46. df.groupby()        : Group by categorical variables
47. df.agg()            : Apply multiple aggregation functions
48. df.crosstab()       : Cross-tabulation (frequency matrix)
49. df.corr()           : Correlation matrix
50. df.cumsum()         : Cumulative sum
"""
print(cheatsheet)

## 🚀 Real Interview Scenario: "Cleaning 50 Million Records"

**Interviewer:** *"You have a customer dataset with 50 million rows and 100 columns. It has missing values, duplicates, and outliers. Your laptop has 16GB of RAM. How do you assess and clean it?"*

### 🧠 The Ideal Answer Framework:
1. **Acknowledge the Constraint:** "50M rows with 100 columns will easily exceed 16GB of RAM if loaded naively into Pandas. I cannot use standard `pd.read_csv()`."
2. **Chunking / Out-of-Core Computing:** "I would use **Dask** or **Polars**, or read the CSV in chunks using Pandas `chunksize=1,000,000`. I would process the data chunk by chunk, aggregating statistics (counts, sums, min/max) iteratively."
3. **Sampling for Profiling:** "To understand the data structure and spot obvious issues, I will take a random 1% sample (500k rows) and load it into Pandas to run `info()`, `describe()`, and visualize distributions."
4. **Handling Duplicates:** "I cannot use `df.drop_duplicates()` in memory. I would sort the data by the primary key using a distributed framework (like PySpark or Dask) and drop duplicates during the sort/shuffle phase."
5. **Handling Missing Data:** "Based on the 1% sample, I'll identify the missing data mechanism. If a column is 80% empty, I'll drop it entirely during the chunked read to save memory. For the rest, I'll calculate global medians chunk-by-chunk and impute."
6. **Validation:** "Finally, I will write `assert` statements or use **Great Expectations** to validate the cleaned chunks before saving them to a Parquet/Snowflake database."

In [ ]:
# ==========================================
# 🏆 MINI PROJECT: RAW TO CLEANED WORKFLOW
# ==========================================
# Let's apply EVERYTHING we learned to the Clinical Trial Dataset.

print("=== PHASE 1: GATHER & PROFILE ===")
# (Assuming patients, treatments, adverse_reactions are loaded from previous cells)
# 1. Create Copies (Golden Rule!)
patients_c = patients.copy()
treatments_c = treatments.copy()

print("=== PHASE 2: ASSESS (Discover & Document) ===")
# We discovered: 
# - patients has missing addresses, inconsistent states, outliers (weight=250).
# - treatments has lowercase names, missing hba1c_change.

print("=== PHASE 3: CLEAN (Define -> Code -> Test) ===")

# STEP 1: Completeness
# Define: Fill missing notes with 'No Notes'
patients_c['notes'] = patients_c['notes'].fillna('No Notes')
# Test
assert patients_c['notes'].isnull().sum() == 0, "Completeness Failed!"

# STEP 2: Tidiness
# Define: Split admission_date to datetime
patients_c['admission_date'] = pd.to_datetime(patients_c['admission_date'])

# STEP 3: Validity
# Define: Ensure weight is float
patients_c['weight_kg'] = pd.to_numeric(patients_c['weight_kg'], errors='coerce')

# STEP 4: Accuracy (Outlier Removal)
# Define: Remove the 250kg outlier using IQR
Q1, Q3 = patients_c['weight_kg'].quantile([0.25, 0.75])
IQR = Q3 - Q1
patients_c = patients_c[patients_c['weight_kg'].between(Q1 - 1.5*IQR, Q3 + 1.5*IQR)]
# Test
assert patients_c['weight_kg'].max() < 150, "Accuracy Failed: Outliers remain!"

# STEP 5: Consistency
# Define: Standardize states
patients_c['state'] = patients_c['state'].replace({'New York': 'NY', 'California': 'CA'})
# Test
assert patients_c['state'].str.len().max() == 2, "Consistency Failed!"

print("=== PHASE 4: VALIDATE & AUDIT ===")
# Hard validation gates
assert patients_c['age'].between(0, 120).all(), "Audit Failed: Invalid Age!"
assert patients_c['weight_kg'].between(30, 200).all(), "Audit Failed: Invalid Weight!"

print("✅ SUCCESS: Raw data transformed to Production-Ready Clean Data!")
print(f"Original Shape: {patients.shape} -> Cleaned Shape: {patients_c.shape}")

## 🎓 Final Summary & Key Takeaways

1. **Assess Before You Clean:** Profile deeply using `nunique`, `memory_usage`, and `value_counts`.
2. **Understand Missingness:** Don't just `fillna()`. Understand MCAR, MAR, MNAR and use the Strategy Matrix.
3. **Detect Outliers Smartly:** Use IQR for skewed data, Z-Score for normal data. Always visualize with Boxplots!
4. **Follow the Golden Order:** Completeness $\rightarrow$ Tidiness $\rightarrow$ Validity $\rightarrow$ Accuracy $\rightarrow$ Consistency.
5. **Define $\rightarrow$ Code $\rightarrow$ Test:** Never write a cleaning script without an `assert` statement to prove it worked.
6. **Scale Up:** For 50M+ rows, abandon standard Pandas. Use Chunking, Dask, Polars, or PySpark.

### 🌟 Fun Fact
> The concept of "Tidy Data" was formalized by Hadley Wickham in 2014. It is heavily inspired by Codd's 3rd Normal Form in relational databases, but optimized specifically for statistical modeling and vectorization in R and Python!

---
**🎉 Congratulations! You have completed the Ultimate Data Assessing & Cleaning Masterclass.**
*You are now ready to tackle any dataset, pass any data cleaning interview, and build production-grade data pipelines!*